# Classifier feature additions with fixed DBSCAN-generated labels

Compare each feature group independently with the current embeddings-only DBSCAN + logistic regression model (validation sample F1 0.910256, macro F1 0.811421). Keep the generated targets, raw weights, per-reason fitting-weight normalization total 157, embedding scaler, C=1000, final cutoff 0.5, and nonempty fallback fixed. No clustering or pseudo-label regeneration.

Choose frequency settings using training documents only. Disease multi-hot minimum counts: **2, 3, 5**. Broad level-0 pathway counts: **1, 5, 30**, each with a missing-annotation indicator; also test that indicator alone. Eligibility text: word unigrams/bigrams, TF-IDF, maximum 2,000 terms, maximum document frequency 95%, minimum document frequencies **3, 50, 100**. Preserve negation words and use sublinear TF and row L2 normalization. Fit vocabulary/IDF on the 1,171 training trials only. Binary features remain 0/1; existing embedding scaling is unchanged. New feature-block weight is fixed at 1.

The text distribution has a long rare tail. A 2,000-term cap already excludes almost all words appearing in fewer than 30 documents, making cutoffs 3, 5, and 10 nearly redundant; the selected cutoffs compare substantially different vocabulary sizes. These settings are declared before consulting new validation results. Evaluate 10 additions plus a reproduced embeddings-only control. Save exact sparse classifier matrices, targets/weights, preprocessing objects, models, and predictions. No Kaggle submission, commit, or tag.

In [1]:
from pathlib import Path
import ast
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
EMBEDDINGS=[f"emb_{i}" for i in range(150)]
DISEASE_MIN_COUNTS=[2,3,5]
PATHWAY_MIN_COUNTS=[1,5,30]
TEXT_MIN_COUNTS=[3,50,100]
TEXT_MAX_FEATURES=2000
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT=DATA/"experiments/classifier_features"
for folder in ["profiles","vocabularies","datasets","models","validation_predictions","submissions"]:
    (OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv")
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
catalog=pd.read_csv(INPUT_DIR/"pathway_catalog.csv")
truth=valid[REASONS].to_numpy(dtype=int)
original_y=train[REASONS].to_numpy(dtype=int)
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
SOURCE=DATA/"experiments/additional_methods"
source_record=json.loads((SOURCE/"submission_manifest.json").read_text())
source_model_path=SOURCE/source_record["model_file"]
source_model=joblib.load(source_model_path)
embedding_scaler=source_model["scaler"]
assert embedding_scaler.n_samples_seen_==157
fixed_frame=pd.read_csv(SOURCE/"datasets/dbscan_m5_q25.csv",float_precision="round_trip")
assert fixed_frame.ID.tolist()==train.ID.tolist()
np.testing.assert_allclose(fixed_frame[EMBEDDINGS].to_numpy(),train[EMBEDDINGS].to_numpy(),rtol=1e-12,atol=1e-15)
Y=fixed_frame[REASONS].to_numpy(dtype=int)
W=fixed_frame[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original_y[known])
np.testing.assert_array_equal(W[known],np.ones((known.sum(),5)))
assert np.array_equal(W>0,Y!=-100)
assert np.isin(Y,[-100,0,1]).all()
targets_frame=fixed_frame[["ID"]+[c for r in REASONS for c in [r,f"weight_{r}"]]].copy()
targets_frame.to_csv(OUT/"datasets/fixed_targets_weights.csv",index=False)
assert int(((W>0).any(axis=1)&~known).sum())==133
protected_paths=[INPUT_DIR/f for f in ["part1_train.csv","part1_val.csv","pathway_catalog.csv"]]+[
    source_model_path,SOURCE/"datasets/dbscan_m5_q25.csv",SOURCE/"labeling_models/dbscan_m5_q25.joblib",
    SOURCE/source_record["submission_file"],SOURCE/"submission_manifest.json"]
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
reference_scaled=embedding_scaler.transform(valid[EMBEDDINGS].to_numpy())
reference_probability=np.column_stack([e.predict_proba(reference_scaled)[:,list(e.classes_).index(1)] for e in source_model["estimators"]])
print("Frozen targets and weights: 157 fully labeled trials +133 trials with generated Efficacy labels only.")


Frozen targets and weights: 157 fully labeled trials +133 trials with generated Efficacy labels only.


In [2]:
def disease_lists(frame):
    def parse(value):
        if pd.isna(value) or not str(value).strip():return []
        values=ast.literal_eval(value)
        assert isinstance(values,list)
        return sorted({str(v).strip().lower() for v in values if str(v).strip()})
    return frame.diseases_cleaned.map(parse).tolist()


def pathway_lists(frame):
    return frame.drug_pathways.fillna("").map(lambda s:sorted({v.strip() for v in str(s).split(";") if v.strip()})).tolist()


def missing_pathways(frame):
    return frame.drug_pathways.fillna("").astype(str).str.strip().eq("").to_numpy(dtype=float)[:,None]

train_diseases=disease_lists(train)
disease_counts=pd.Series([v for row in train_diseases for v in row]).value_counts().rename_axis("disease").rename("training_trials")
disease_known_counts=pd.Series([v for row in np.array(train_diseases,dtype=object)[known] for v in row]).value_counts()
disease_profile=disease_counts.to_frame()
disease_profile["known_label_trials"]=disease_known_counts.reindex(disease_counts.index,fill_value=0).astype(int)
disease_profile.to_csv(OUT/"profiles/disease_frequencies.csv")
train_pathways=pathway_lists(train)
all_catalog_ids=set(catalog.reactome_id)
assert set(v for row in train_pathways for v in row)<=all_catalog_ids
broad_catalog=catalog.loc[catalog.pathway_level==0].sort_values("reactome_id")
assert len(broad_catalog)==29
pathway_profile=broad_catalog[["reactome_id","pathway_name","pathway_level"]].copy()
pathway_profile["training_trials"]=[sum(pid in row for row in train_pathways) for pid in pathway_profile.reactome_id]
pathway_profile["known_label_trials"]=[sum(pid in train_pathways[i] for i in np.flatnonzero(known)) for pid in pathway_profile.reactome_id]
pathway_profile.to_csv(OUT/"profiles/broad_pathway_frequencies.csv",index=False)
count_vectorizer=CountVectorizer(lowercase=True,strip_accents="unicode",ngram_range=(1,2),stop_words=None)
counts=count_vectorizer.fit_transform(train.criteria.fillna(""))
text_df=np.asarray((counts>0).sum(axis=0)).ravel()
text_profile=pd.DataFrame({"term":count_vectorizer.get_feature_names_out(),"training_trials":text_df,
                           "total_occurrences":np.asarray(counts.sum(axis=0)).ravel()})
text_profile.to_csv(OUT/"profiles/criteria_term_frequencies.csv",index=False)
profile_rows=[]
for minimum in [1,2,3,5,10,20,30,50,75,100]:
    profile_rows.append({"minimum_training_trials":minimum,"disease_features":int((disease_counts>=minimum).sum()),
                         "broad_pathway_features":int((pathway_profile.training_trials>=minimum).sum()),
                         "criteria_terms_before_cap":int(((text_df>=minimum)&(text_df<=0.95*len(train))).sum())})
pd.DataFrame(profile_rows).to_csv(OUT/"profiles/frequency_cutoff_sizes.csv",index=False)
print("Disease vocabulary counts:",{m:int((disease_counts>=m).sum()) for m in DISEASE_MIN_COUNTS})
print("Broad pathway counts:",{m:int((pathway_profile.training_trials>=m).sum()) for m in PATHWAY_MIN_COUNTS})
print("Criteria raw vocabulary:",len(text_df),"terms in one trial:",int((text_df==1).sum()))
del counts,count_vectorizer


Disease vocabulary counts: {2: 249, 3: 145, 5: 71}
Broad pathway counts: {1: 29, 5: 27, 30: 26}
Criteria raw vocabulary: 104482 terms in one trial: 67965


In [3]:
def make_block(kind,minimum=None):
    if kind=="none":return {"kind":"none","feature_names":[]}
    if kind=="diseases":
        classes=sorted(disease_counts[disease_counts>=minimum].index.tolist())
        encoder=MultiLabelBinarizer(classes=classes,sparse_output=True)
        encoder.fit(train_diseases)
        return {"kind":kind,"minimum_training_trials":minimum,"encoder":encoder,
                "feature_names":[f"disease__{v}" for v in classes]}
    if kind=="pathways":
        classes=pathway_profile.loc[pathway_profile.training_trials>=minimum,"reactome_id"].tolist()
        encoder=MultiLabelBinarizer(classes=classes,sparse_output=True)
        encoder.fit(train_pathways)
        return {"kind":kind,"minimum_training_trials":minimum,"encoder":encoder,
                "feature_names":[f"pathway__{v}" for v in classes]+["pathway_annotation_missing"]}
    if kind=="missing_only":return {"kind":kind,"feature_names":["pathway_annotation_missing"]}
    if kind=="criteria":
        vectorizer=TfidfVectorizer(lowercase=True,strip_accents="unicode",ngram_range=(1,2),stop_words=None,
                    min_df=minimum,max_df=0.95,max_features=TEXT_MAX_FEATURES,sublinear_tf=True,norm="l2")
        vectorizer.fit(train.criteria.fillna(""))
        return {"kind":kind,"minimum_training_trials":minimum,"vectorizer":vectorizer,
                "feature_names":[f"criteria__{v}" for v in vectorizer.get_feature_names_out()]}
    raise ValueError(kind)


def transform_block(block,frame):
    kind=block["kind"]
    if kind=="none":return sparse.csr_matrix((len(frame),0))
    if kind=="missing_only":return sparse.csr_matrix(missing_pathways(frame))
    if kind=="criteria":return block["vectorizer"].transform(frame.criteria.fillna("")).tocsr()
    values=disease_lists(frame) if kind=="diseases" else pathway_lists(frame)
    allowed=set(block["encoder"].classes_)
    result=block["encoder"].transform([[v for v in row if v in allowed] for row in values]).astype(float).tocsr()
    if kind=="pathways":result=sparse.hstack([result,sparse.csr_matrix(missing_pathways(frame))],format="csr")
    assert result.shape[1]==len(block["feature_names"])
    assert np.isin(result.data,[0,1]).all()
    return result


def transform_model(model,frame):
    embeddings=model["embedding_scaler"].transform(frame[model["embedding_features"]].to_numpy())
    if model["block"]["kind"]=="none":return embeddings
    return sparse.hstack([sparse.csr_matrix(embeddings),transform_block(model["block"],frame)],format="csr")


def predict_model(model,frame):
    matrix=transform_model(model,frame)
    return np.column_stack([est.predict_proba(matrix)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int)
    empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty

reference_prediction,_=label_predictions(reference_probability)
reference_sample=f1_score(truth,reference_prediction,average="samples",zero_division=0)
reference_macro=f1_score(truth,reference_prediction,average="macro",zero_division=0)
np.testing.assert_allclose(reference_sample,source_record["validation_sample_f1"],atol=1e-14)
np.testing.assert_allclose(reference_macro,source_record["validation_macro_f1"],atol=1e-14)
ROWS=[];MODELS={};FEATURE_HASHES={}


def run_configuration(kind,minimum=None):
    name="embeddings_only" if kind=="none" else ("pathway_missing_only" if kind=="missing_only" else f"{kind}_min{minimum}")
    block=make_block(kind,minimum)
    model={"kind":"embeddings_plus_features_logistic","C":1000.0,"embedding_scaler":embedding_scaler,
           "embedding_features":EMBEDDINGS,"reasons":REASONS,"block":block,"estimators":[],
           "classifier_weight_normalization_total":157,"cutoff":0.5,"nonempty_fallback":True,
           "source_model_file":str(source_model_path.relative_to(DATA)),
           "source_targets_file":"datasets/fixed_targets_weights.csv","feature_block_weight":1.0}
    matrix=transform_model(model,train)
    saved_matrix=sparse.csr_matrix(matrix)
    sparse.save_npz(OUT/"datasets"/f"{name}.npz",saved_matrix)
    feature_names=EMBEDDINGS+block["feature_names"]
    (OUT/"datasets"/f"{name}_columns.json").write_text(json.dumps(feature_names,indent=2)+"\n")
    assert matrix.shape==(len(train),len(feature_names)) and np.isfinite(saved_matrix.data).all()
    assert len(set(feature_names))==len(feature_names)
    with threadpool_limits(limits=1):
        for j in range(5):
            estimator=clone(source_model["estimators"][j]);included=W[:,j]>0
            assert estimator.C==1000.0 and included[known].all()
            normalized=W[included,j]*157/W[included,j].sum()
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(matrix[included],Y[included,j],sample_weight=normalized)
            model["estimators"].append(estimator)
    probabilities=predict_model(model,valid)
    if kind=="none":np.testing.assert_allclose(probabilities,reference_probability,rtol=1e-8,atol=1e-10)
    predicted,empty=label_predictions(probabilities)
    metrics={"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
             "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
             **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))}
    # These are apparent training scores on the original, fully labeled trials only.
    known_probability=predict_model(model,train.loc[known])
    known_prediction,_=label_predictions(known_probability)
    known_train_f1=f1_score(original_y[known],known_prediction,average="samples",zero_division=0)
    feature_hash=hashlib.sha256(saved_matrix.data.tobytes()+saved_matrix.indices.tobytes()+saved_matrix.indptr.tobytes()+str(saved_matrix.shape).encode()).hexdigest()
    FEATURE_HASHES[name]=feature_hash
    ROWS.append({"experiment":name,"feature_group":kind,"minimum_training_trials":minimum,
                 "additional_features":len(block["feature_names"]),"total_features":len(feature_names),
                 "training_preprocessing_rows":1171,"known_training_sample_f1":known_train_f1,
                 **metrics,"fallback_trials":int(empty.sum()),"changed_validation_trials":int((predicted!=reference_prediction).any(axis=1).sum()),
                 "delta_sample_f1":metrics["sample_f1"]-reference_sample,"delta_macro_f1":metrics["macro_f1"]-reference_macro,
                 "matrix_sha256":feature_hash,"kaggle_score":np.nan})
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=predicted[:,j]
    frame["used_fallback"]=empty
    frame["trial_f1"]=np.divide(2*(truth*predicted).sum(axis=1),truth.sum(axis=1)+predicted.sum(axis=1))
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3);MODELS[name]=model
    if kind in ["diseases","pathways"]:
        key="disease" if kind=="diseases" else "reactome_id"
        vocabulary=pd.DataFrame({key:block["encoder"].classes_})
        if kind=="diseases":vocabulary=vocabulary.merge(disease_profile.reset_index(),on=key,how="left")
        else:vocabulary=vocabulary.merge(pathway_profile,on=key,how="left")
        vocabulary.to_csv(OUT/"vocabularies"/f"{name}.csv",index=False)
    elif kind=="criteria":
        terms=block["vectorizer"].get_feature_names_out()
        vocabulary=pd.DataFrame({"term":terms,"idf":block["vectorizer"].idf_}).merge(text_profile,on="term",how="left")
        vocabulary.to_csv(OUT/"vocabularies"/f"{name}.csv",index=False)
    print(f"{name}: +{len(block['feature_names'])} features, sample F1={metrics['sample_f1']:.6f}, "
          f"macro F1={metrics['macro_f1']:.6f}, change={metrics['sample_f1']-reference_sample:+.6f}",flush=True)

run_configuration("none")


embeddings_only: +0 features, sample F1=0.910256, macro F1=0.811421, change=+0.000000


In [4]:
for minimum in DISEASE_MIN_COUNTS:run_configuration("diseases",minimum)


diseases_min2: +249 features, sample F1=0.910256, macro F1=0.811421, change=+0.000000


diseases_min3: +145 features, sample F1=0.910256, macro F1=0.811421, change=+0.000000


diseases_min5: +71 features, sample F1=0.910256, macro F1=0.811421, change=+0.000000


In [5]:
run_configuration("missing_only")
for minimum in PATHWAY_MIN_COUNTS:run_configuration("pathways",minimum)


pathway_missing_only: +1 features, sample F1=0.905983, macro F1=0.805106, change=-0.004274


pathways_min1: +30 features, sample F1=0.873932, macro F1=0.760207, change=-0.036325


pathways_min5: +28 features, sample F1=0.873932, macro F1=0.760207, change=-0.036325


pathways_min30: +27 features, sample F1=0.873932, macro F1=0.760207, change=-0.036325


In [6]:
for minimum in TEXT_MIN_COUNTS:run_configuration("criteria",minimum)


criteria_min3: +2000 features, sample F1=0.910256, macro F1=0.811421, change=+0.000000


criteria_min50: +1533 features, sample F1=0.897436, macro F1=0.801106, change=-0.012821


criteria_min100: +776 features, sample F1=0.893162, macro F1=0.792111, change=-0.017094


In [7]:
comparison=pd.DataFrame(ROWS)
assert len(comparison)==11
comparison.to_csv(OUT/"comparison.csv",index=False)
ranked=comparison.assign(is_reference=comparison.experiment.eq("embeddings_only")).sort_values(
    ["sample_f1","is_reference","additional_features","experiment"],ascending=[False,False,True,True],kind="stable")
best=ranked.iloc[0]
best_by_group=ranked.groupby("feature_group",sort=False).head(1)
best_by_group.to_csv(OUT/"best_by_group.csv",index=False)
print("Best overall:",best.experiment,"sample F1:",best.sample_f1)
display(comparison[["experiment","additional_features","sample_f1","macro_f1","delta_sample_f1","fallback_trials"]].round(6))
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid)
np.testing.assert_allclose(selected_probability,predict_model(MODELS[best.experiment],valid),rtol=1e-12,atol=1e-12)
# Every saved matrix must exactly reproduce its model's training preprocessing.
for name,model in MODELS.items():
    persisted=sparse.load_npz(OUT/"datasets"/f"{name}.npz")
    expected=sparse.csr_matrix(transform_model(model,train))
    difference=persisted-expected
    assert not difference.nnz or abs(difference.data).max()<1e-12
    assert persisted.shape[0]==len(targets_frame)
    frame=pd.read_csv(OUT/"validation_predictions"/f"{name}.csv")
    predicted=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    expected_prediction=label_predictions(predict_model(joblib.load(OUT/"models"/f"{name}.joblib"),valid))[0]
    np.testing.assert_array_equal(predicted,expected_prediction)
    row=comparison.loc[comparison.experiment==name].iloc[0]
    np.testing.assert_allclose(f1_score(truth,predicted,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
# Independently refit the selected model from its saved matrix and raw targets/weights.
replay_matrix=sparse.load_npz(OUT/"datasets"/f"{best.experiment}.npz")
if selected["block"]["kind"]=="none":replay_matrix=replay_matrix.toarray()
fixed_replay=pd.read_csv(OUT/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed_replay.ID.tolist()==train.ID.tolist()
np.testing.assert_array_equal(fixed_replay[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(fixed_replay[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=fixed_replay[r].to_numpy(dtype=int);w=fixed_replay[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(replay_matrix[included],y[included],sample_weight=w[included]*157/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid))[0],label_predictions(selected_probability)[0])
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest
print("All saved matrices, reloaded predictions, fixed targets/weights, reference reproduction, and independent winner refit passed.")


Best overall: embeddings_only sample F1: 0.9102564102564102


,experiment,additional_features,sample_f1,macro_f1,delta_sample_f1,fallback_trials
0,embeddings_only,0,0.910256,0.811421,0.000000,4
1,diseases_min2,249,0.910256,0.811421,0.000000,4
2,diseases_min3,145,0.910256,0.811421,0.000000,4
3,diseases_min5,71,0.910256,0.811421,0.000000,4
4,pathway_missing_only,1,0.905983,0.805106,-0.004274,4
5,pathways_min1,30,0.873932,0.760207,-0.036325,8
6,pathways_min5,28,0.873932,0.760207,-0.036325,8
7,pathways_min30,27,0.873932,0.760207,-0.036325,8
8,criteria_min3,2000,0.910256,0.811421,0.000000,4
9,criteria_min50,1533,0.897436,0.801106,-0.012821,4


All saved matrices, reloaded predictions, fixed targets/weights, reference reproduction, and independent winner refit passed.


In [8]:
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    test_probability=predict_model(selected,test)
    prediction,empty=label_predictions(test_probability)
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_logistic_C1000_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"reference_experiment":source_record["experiment"],"reference_model_file":str(source_model_path.relative_to(DATA)),
          "reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "features_changed_in":"classifier only","generated_labels_changed":False,"generated_weights_changed":False,
          "embedding_scaler_fit_rows":157,"new_preprocessing_fit_rows":1171,"preprocessing_uses_validation":False,
          "disease_minimum_training_counts":DISEASE_MIN_COUNTS,"pathway_minimum_training_counts":PATHWAY_MIN_COUNTS,
          "pathway_level":0,"pathway_missing_indicator":True,"pathway_missing_only_control":True,
          "text_minimum_training_counts":TEXT_MIN_COUNTS,"text_max_features":TEXT_MAX_FEATURES,
          "text_ngram_range":[1,2],"text_max_df":0.95,"text_stop_words":None,"text_sublinear_tf":True,"text_norm":"l2",
          "binary_feature_scaling":"0/1; no centering or variance scaling","new_feature_block_weight":1.0,
          "classifier_C":1000,"fitting_weight_total_per_reason":157,"prediction_cutoff":0.5,"nonempty_fallback":True,
          "validation_rows":78,"configurations":len(comparison),"selection":"sample-wise F1; ties favor reference, then fewer added features",
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "candidate":candidate,"protected_input_sha256":protected_hashes,"matrix_sha256":FEATURE_HASHES,
          "saved_dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*')) if p.is_file()},
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"scipy":scipy.__version__,"numpy":np.__version__,"pandas":pd.__version__},
          "new_kaggle_scores":None,"reported_best_kaggle":0.87393}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Classifier feature additions with frozen DBSCAN pseudo-labels","",
       f"Reference: sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       "Each feature group is added separately to the same 150 embeddings. No clustering, target, or generated-weight changes. "
       "The same 157 known-label trials and 133 trials with generated Efficacy labels are used; each reason retains its own targets/weights.","",
       "## Training-only frequency inspection","",
       f"Diseases: {len(disease_counts)} distinct normalized names; minimum counts 2/3/5 retain "
       f"{int((disease_counts>=2).sum())}/{int((disease_counts>=3).sum())}/{int((disease_counts>=5).sum())} indicators. "
       "Counts are numbers of distinct training trials, not repeated mentions.","",
       "Pathways: 29 broad level-0 categories. Count thresholds 1/5/30 retain 29/27/26 categories; each adds a missing-annotation indicator. "
       "That indicator is also evaluated alone. Missing annotation does not imply no drug or no biological activity.","",
       f"Criteria: {len(text_profile)} unigram/bigram terms, including {int((text_profile.training_trials==1).sum())} occurring in one trial. "
       "With a 2,000-term cap, low cutoffs such as 3/5/10 would select nearly identical vocabularies. "
       "Minimum document counts 3/50/100 were chosen to test progressively stronger filtering. All criteria vocabularies use "
       "maximum DF 95%, sublinear TF, row L2 normalization, and preserved negation words. No stop-word list is used.","",
       "## Validation results","",
       "| Configuration | Added features | Sample F1 | Macro F1 | Change in sample F1 | Fallback trials |",
       "|---|---:|---:|---:|---:|---:|"]
for row in comparison.itertuples(index=False):
    lines.append(f"| {row.experiment} | {row.additional_features} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_sample_f1:+.6f} | {row.fallback_trials} |")
lines.extend(["",f"Selected: **{best.experiment}**, sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Controls and interpretation","",
       "The embedding scaler stays frozen from the 157 originally labeled trials. New feature vocabularies and IDF are "
       "fit only on all 1,171 training trials, using no validation/test documents or labels. Binary features stay 0/1; "
       "TF-IDF uses L2 row normalization. New blocks have weight 1. C=1000, normalized fitting-weight total 157, "
       "final cutoff 0.5, and highest-probability fallback stay fixed. No feature combinations are evaluated in this round.","",
       "All saved sparse feature matrices reproduce their preprocessing. Saved models reload with matching predictions, "
       "the embeddings-only control reproduces the original probabilities, and the selected model independently refits "
       "from its persisted matrix and targets/weights with matching probabilities. Protected original artifacts retain their hashes.","",
       "Performance differences reflect these specific encodings/filtering settings at fixed C and feature-block scale; "
       "a negative result does not establish that a raw feature can never help. This is exploratory selection on the same "
       "78 validation trials, not an independent test estimate.","",
       "`profiles/` preserves frequency distributions and counts; `vocabularies/` records accepted terms/categories. "
       "`datasets/` saves each exact sparse classifier matrix with ordered feature names, and the shared trial IDs/targets/weights. "
       "`models/` saves native sklearn preprocessing and estimators; `validation_predictions/` saves per-trial outputs. "
       "`comparison.csv`, `best_by_group.csv`, and `settings.json` preserve scores, settings, and checksums.","",
       "No Kaggle submission, commit, or tag was made.",""])
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No addition improved the primary metric; retain the embeddings-only DBSCAN model as the leader.")
summary="\n".join(lines)+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(summary[:4500])


# Classifier feature additions with frozen DBSCAN pseudo-labels

Reference: sample F1 **0.910256**, macro F1 **0.811421**. Each feature group is added separately to the same 150 embeddings. No clustering, target, or generated-weight changes. The same 157 known-label trials and 133 trials with generated Efficacy labels are used; each reason retains its own targets/weights.

## Training-only frequency inspection

Diseases: 879 distinct normalized names; minimum counts 2/3/5 retain 249/145/71 indicators. Counts are numbers of distinct training trials, not repeated mentions.

Pathways: 29 broad level-0 categories. Count thresholds 1/5/30 retain 29/27/26 categories; each adds a missing-annotation indicator. That indicator is also evaluated alone. Missing annotation does not imply no drug or no biological activity.

Criteria: 104482 unigram/bigram terms, including 67965 occurring in one trial. With a 2,000-term cap, low cutoffs such as 3/5/10 would select nearly identical vocabularies. Minim